# Chapter 7 — Star clusters as a test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/the-cannon-tutorial/blob/main/notebooks/07_clusters.ipynb)

**Goal.** Use star clusters as an external check of the labels. The stars of a cluster formed together: they
share an age, a distance and — to a good approximation for iron in open clusters — a chemical composition. Their
giants must therefore lie on one sequence in the Kiel diagram and have the same [Fe/H]. N15 *trained* on
cluster stars; we never did (the training set excludes all members), so the clusters are an independent test.

**You will learn**
1. to compare cluster mean metallicities between two methods, across clusters (a constant predictor would fail);
2. to estimate the spread of a label within a cluster and compare it with the measurement errors;
3. where the test stops being a test: globular clusters below the training range.

Caveats: APOGEE's `member` flag is a preliminary membership (position, radial velocity, proper motion), not a
vetted probability; and globular clusters are *not* homogeneous in light elements (C, N, O, Na, Mg, Al vary from
star to star), so we only use [Fe/H] for them.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of APOGEE DR17 and Gaia DR3, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the query that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/the-cannon-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name> (.parquet table or .npz arrays) from the data release if needed, check its SHA-256, read it.

    A .parquet file is returned as a pandas DataFrame, an .npz file as a dict of numpy arrays.
    """
    ext = '.npz' if name.startswith(('spectra_', 'rvs_')) else '.parquet'
    path = cache / f'{name}{ext}'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same data
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    if ext == '.parquet':
        return pd.read_parquet(path)
    with np.load(path, allow_pickle=False) as f:   # allow_pickle=False: arrays only, no code can be loaded
        return {k: f[k] for k in f.files}
# --- Spectra and continuum normalisation (as in Chapters 0 and 3) ----------------------------------------
NPIX = 8575
wave = 10**(4.179 + 6e-6 * np.arange(NPIX))               # apStar vacuum wavelength grid [Angstrom]
CHIPS = [(15150, 15800), (15865, 16425), (16480, 16950)]  # the three detectors [A], inside the edges of Chapter 0
BAD_BITS = sum(1 << b for b in [0, 1, 2, 3, 4, 5, 6, 7, 12])   # pixels given zero weight (Chapter 0)


def prepare(sp, rows=slice(None)):
    """flux and err (float64) of the spectra file sp; err = inf where there are no data, a BAD_BITS flag, or
    outside CHIPS (the detector edges, where only some spectra have data)."""
    flux = sp['flux'][rows].astype(float)
    err = sp['err'][rows].astype(float)
    on_chip = np.any([(wave > lo) & (wave < hi) for lo, hi in CHIPS], axis=0)   # pixels at the detector edges and
    bad = ((sp['mask'][rows] & BAD_BITS) > 0) | ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~on_chip  # gaps: no weight
    return np.where(bad, 0.0, flux), np.where(bad, np.inf, err)


def cheb_normalise(flux, err, cont, order=2):
    """Divide by a Chebyshev polynomial of the given order fitted, per detector, to the continuum pixels `cont`
    (boolean, NPIX) with weights 1/err^2 (N15 sect. 2.3 and 5.3). Returns normalised flux and error;
    pixels outside the detectors get err = inf."""
    fn, en = np.zeros_like(flux), np.full_like(err, np.inf)
    for lo, hi in CHIPS:
        c = (wave > lo) & (wave < hi)
        t = (wave[c] - (lo + hi) / 2) / ((hi - lo) / 2)          # wavelength mapped to [-1, 1]
        B = np.polynomial.chebyshev.chebvander(t, order)         # (pixels, order+1) basis
        use = cont[c]
        w = 1 / err[:, c][:, use]**2                             # (N, continuum pixels); 0 for flagged pixels
        # one small weighted least-squares problem per star: (B^T W B) a = B^T W f
        BtWB = np.einsum('pi,np,pj->nij', B[use], w, B[use])
        BtWf = np.einsum('pi,np->ni', B[use], w * flux[:, c][:, use])
        a = np.linalg.solve(BtWB, BtWf[:, :, None])[:, :, 0]     # (N, order+1); b as a stack of column vectors
        cont_fit = a @ B.T                                       # (N, pixels of this detector)
        fn[:, c] = flux[:, c] / cont_fit
        en[:, c] = err[:, c] / cont_fit
    return fn, en
# --- The Cannon (as written in Chapter 2) -----------------------------------------------------------------
from scipy.optimize import least_squares


def vectorize(x):
    """Quadratic label vector v(x) = [1, x_k, x_k x_l (k <= l)] for scaled labels x of shape (N, K): (N, D)."""
    x = np.atleast_2d(x)
    k, l = np.triu_indices(x.shape[1])
    return np.hstack([np.ones((len(x), 1)), x, x[:, k] * x[:, l]])


def dvectorize(x):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    K = len(x)
    k, l = np.triu_indices(K)
    J = np.zeros((1 + K + len(k), K))
    J[1:1 + K] = np.eye(K)
    for i, (a, b) in enumerate(zip(k, l)):
        J[1 + K + i, a] += x[b]
        J[1 + K + i, b] += x[a]
    return J


def fit_theta(V, flux, var):
    """Weighted least squares at every pixel: theta (P, D) and its covariance (P, D, D); weights 1/var (N, P).
    Zero-weight entries (var = inf) drop out of the sums."""
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)                       # NaN flux at zero weight must not poison the sums
    N, Dv = V.shape
    A = ((V[:, :, None] * V[:, None, :]).reshape(N, Dv * Dv).T @ w).T.reshape(-1, Dv, Dv)
    b = (V.T @ (w * flux)).T
    cov = np.linalg.pinv(A)                                 # pinv: pixels without data give theta = 0, not an error
    return np.einsum('pde,pe->pd', cov, b), cov


S_GRID = np.r_[0, np.logspace(-4, 0, 41)]                  # 0, then 1e-4 ... 1 in steps of 0.1 dex


def fit_scatter(resid, err2):
    """Maximum-likelihood scatter s per pixel for fixed residuals: best value on the grid S_GRID, refined by a
    parabola in log s (Chapter 2). Zero-weight entries (err2 = inf) are ignored."""
    use = np.isfinite(err2) & np.isfinite(resid)
    r2 = np.where(use, resid**2, 0).astype(np.float32)
    e2 = np.where(use, err2, 1e30).astype(np.float32)        # unused entries: a constant term, whatever s is
    nll = np.array([np.sum(r2 / (e2 + s**2) + np.log(e2 + s**2), axis=0) for s in S_GRID.astype(np.float32)])
    j = np.clip(np.argmin(nll, axis=0), 2, len(S_GRID) - 2)
    p = np.arange(nll.shape[1])
    a, b, c = nll[j - 1, p], nll[j, p], nll[j + 1, p]
    shift = 0.5 * (a - c) / np.where(a - 2 * b + c > 0, a - 2 * b + c, np.inf)
    return np.where(np.argmin(nll, axis=0) == 0, 0.0, S_GRID[j] * np.exp(shift * np.log(S_GRID[2] / S_GRID[1])))


def train(x, flux, err, n_iter=3):
    """Training step. x: scaled labels (N, K); flux, err: (N, P) (err = inf for pixels to ignore).
    Returns theta (P, D), its covariance (P, D, D) and the scatter s (P,)."""
    V, err2 = vectorize(x), err**2
    s = np.zeros(flux.shape[1])
    for _ in range(n_iter):
        theta, cov = fit_theta(V, flux, err2 + s**2)
        s = fit_scatter(flux - V @ theta.T, err2)
    theta, cov = fit_theta(V, flux, err2 + s**2)
    return theta, cov, s


def test(flux, err, theta, s, x0=None):
    """Test step for one spectrum: scaled labels (K,), covariance (K, K), chi^2, number of pixels used."""
    K = int(round((np.sqrt(8 * theta.shape[1] + 1) - 3) / 2))   # invert D = 1 + K + K(K+1)/2
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize(x)[0]) * w
    jac = lambda x: -(th @ dvectorize(x)) * w[:, None]
    r = least_squares(res, np.zeros(K) if x0 is None else x0, jac=jac, method='lm')
    return r.x, np.linalg.inv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum()


def train_clean(x, flux, err, cut=3.0):
    """Train, drop the training stars whose mean squared residual per pixel (in units of sigma^2 + s^2) exceeds
    `cut`, and train again (Chapter 4). Returns theta, its covariance, s, and the boolean mask of stars kept."""
    theta, cov, s = train(x, flux, err)
    z2 = (flux - vectorize(x) @ theta.T)**2 / (err**2 + s**2)
    z2 = np.where(np.isfinite(err) & (theta[:, 0] != 0), z2, np.nan)
    keep = np.nanmean(z2, axis=1) < cut
    theta, cov, s = train(x[keep], flux[keep], err[keep])
    return theta, cov, s, keep

labels = load('labels')
cont = load('continuum_pixels')['continuum'].to_numpy()
NAMES, TEX = ['teff', 'logg', 'fe_h'], [r'$T_{\rm eff}$', r'$\log g$', '[Fe/H]']
tr = labels[labels['sample'] == 'train'].reset_index(drop=True)
L = tr[NAMES].to_numpy()
pv = np.median(L, axis=0)
sc = (np.percentile(L, 97.5, axis=0) - np.percentile(L, 2.5, axis=0)) / 2
th, _, s, _ = train_clean((L - pv) / sc, *cheb_normalise(*prepare(load('spectra_train')), cont))
cl = labels[labels['sample'] == 'clusters'].reset_index(drop=True)
fc, ec = cheb_normalise(*prepare(load('spectra_clusters')), cont)
out = [test(fc[n], ec[n], th, s) for n in range(len(cl))]
can = np.array([o[0] for o in out]) * sc + pv
cl['teff_c'], cl['logg_c'], cl['fe_h_c'] = can.T
cl['chi2nu'] = [o[2] / (o[3] - 3) for o in out]
OPEN = ['M67', 'N6819', 'N188', 'N7789', 'N2420', 'N6791', 'N2158']
cl['type'] = np.where(cl.member.isin(OPEN), 'open', 'globular')
print(cl.groupby('member').size().sort_values(ascending=False).to_string())

## 7.1 Kiel diagrams

In [ ]:
order = cl.groupby('member').fe_h.median().sort_values(ascending=False).index
fig, ax = plt.subplots(3, 6, figsize=(17, 9), sharex=True, sharey=True)
for a, m in zip(ax.flat, order):
    d = cl[cl.member == m]
    a.plot(d.teff, d.logg, 'o', mfc='none', color='0.5', ms=4, label='ASPCAP')
    a.plot(d.teff_c, d.logg_c, '.', color='C3', ms=5, label='Cannon')
    a.set_title(f'{m} ({d.type.iloc[0]}), [Fe/H] ≈ {d.fe_h.median():.2f}', fontsize=9)
for a in ax.flat[len(order):]:
    a.axis('off')
ax[0, 0].set(xlim=(5400, 3700), ylim=(4, 0))
ax[0, 0].legend(fontsize=7)
for a in ax[:, 0]:
    a.set_ylabel(r'$\log g$')
for a in ax[-1]:
    a.set_xlabel(r'$T_{\rm eff}$ [K]')
plt.tight_layout()

**Figure 7.1.** Kiel diagrams of the 17 clusters, ordered by metallicity (title: cluster, type, median ASPCAP
[Fe/H]): ASPCAP labels (open grey circles) and The Cannon's (red dots) for the same stars. For the open
clusters and the metal-rich globulars the two agree and trace narrow giant branches (and red clumps). For the
most metal-poor globulars (bottom row: M53, M92, M15, [Fe/H] < −1.5, outside the training range) The Cannon's
giants are displaced to higher gravity, by ≈ 0.7 dex in $\log g$ at the same $T_{\rm eff}$ — Chapter 5's
extrapolation problem (the metal-poor field giants showed the same). In M13 the shift has started already.

## 7.2 Mean metallicities across clusters

The mean [Fe/H] of each cluster from The Cannon against that from ASPCAP. Agreement for one cluster is easy (even
a model that returned the training mean for every star would agree at [Fe/H] ≈ −0.3); agreement over a range of
clusters, i.e. a slope of 1, is a real test.

In [ ]:
g = cl.groupby('member')
tab = pd.DataFrame({'n': g.size(), 'type': g.type.first(),
                    'feh_aspcap': g.fe_h.mean(), 'feh_cannon': g.fe_h_c.mean(),
                    'std_aspcap': g.fe_h.std(), 'std_cannon': g.fe_h_c.std()})
tab['err_cannon'] = tab.std_cannon / np.sqrt(tab.n)
inside = tab.feh_aspcap > L[:, 2].min()                     # cluster mean inside the training range of [Fe/H]
print(tab.round(3).sort_values('feh_aspcap', ascending=False).to_string())
p = np.polyfit(tab.feh_aspcap[inside], tab.feh_cannon[inside], 1, w=1 / tab.err_cannon[inside].clip(0.01))
fig, ax = plt.subplots(figsize=(6.5, 5))
for t, mk in [('open', 'o'), ('globular', 's')]:
    d = tab[(tab.type == t)]
    ax.errorbar(d.feh_aspcap, d.feh_cannon, d.err_cannon, fmt=mk, mfc='none', label=f'{t} clusters', capsize=2)
xx = np.array([-2.4, 0.5])
ax.plot(xx, xx, 'k-', lw=0.8, label='identity')
ax.plot(xx, np.polyval(p, xx), 'C3--', lw=1, label=f'fit inside training range: slope {p[0]:.2f}')
ax.axvspan(-2.6, L[:, 2].min(), color='0.9', zorder=-1)
ax.set(xlabel='mean ASPCAP [Fe/H]', ylabel='mean Cannon [Fe/H]', xlim=(-2.5, 0.5), ylim=(-2.5, 0.5))
ax.legend(fontsize=8)
plt.tight_layout()
print(f'slope {p[0]:.3f}, offset at [Fe/H] = 0: {p[1]:+.3f}')
# Test: inside the training range The Cannon reproduces the differences between clusters (slope 0.9-1.1).
assert 0.9 < p[0] < 1.1

**Figure 7.2.** Mean [Fe/H] of each cluster from The Cannon against ASPCAP, with the standard error of the
Cannon mean; circles: open clusters, squares: globular clusters. The dashed line is a weighted fit to the
clusters inside the training range of [Fe/H] (white); the grey band is below it. Inside the range the points
follow the identity (slope 0.98); below it the three most metal-poor globular clusters (M53, M92, M15, ASPCAP
[Fe/H] ≈ −1.9 to −2.3) all come out at ≈ −1.5, the edge of the training set, as the metal-poor field giants of
Chapter 5 did.

## 7.3 Spread within clusters

If the stars of an open cluster have the same [Fe/H], the spread of their measured [Fe/H] is the measurement
error. Compare the standard deviation within each cluster for the two methods, and look for a trend with
temperature, which would betray a systematic error that depends on the stellar type.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for t, mk in [('open', 'o'), ('globular', 's')]:
    d = tab[(tab.type == t) & inside]
    ax[0].plot(d.std_aspcap, d.std_cannon, mk, mfc='none', label=f'{t} (inside training range)')
ax[0].plot([0, 0.15], [0, 0.15], 'k-', lw=0.8)
ax[0].set(xlabel='std of ASPCAP [Fe/H] within cluster', ylabel='std of Cannon [Fe/H] within cluster')
ax[0].legend(fontsize=8)
oc = cl[cl.type == 'open'].copy()
oc['dfe'] = oc.fe_h_c - oc.groupby('member').fe_h_c.transform('mean')   # deviation from the cluster mean
oc['dfe_a'] = oc.fe_h - oc.groupby('member').fe_h.transform('mean')
ax[1].plot(oc.teff_c, oc.dfe, '.', color='C3', label='Cannon')
ax[1].plot(oc.teff, oc.dfe_a, 'o', mfc='none', color='0.5', ms=4, label='ASPCAP')
slope_c = np.polyfit(oc.teff_c, oc.dfe, 1)[0] * 1000
slope_a = np.polyfit(oc.teff, oc.dfe_a, 1)[0] * 1000
ax[1].axhline(0, color='k', lw=0.8)
ax[1].set(xlabel=r'$T_{\rm eff}$ [K]', ylabel='[Fe/H] − cluster mean [dex]',
          title=f'open clusters; slopes per 1000 K: Cannon {slope_c:+.3f}, ASPCAP {slope_a:+.3f}')
ax[1].legend(fontsize=8)
plt.tight_layout()
oc_std = tab[tab.type == 'open'].std_cannon
print(f'open clusters: median within-cluster std of [Fe/H]: Cannon {oc_std.median():.3f}, ASPCAP {tab[tab.type == "open"].std_aspcap.median():.3f} dex')
# Test: open clusters are homogeneous in Cannon [Fe/H] to better than 0.05 dex (median std), comparable to ASPCAP.
assert oc_std.median() < 0.05 and oc_std.median() < 1.5 * tab[tab.type == 'open'].std_aspcap.median()

**Figure 7.3.** Left: standard deviation of [Fe/H] within each cluster, The Cannon against ASPCAP (clusters
inside the training range; the line is equality). Right: deviation of each open-cluster star from its cluster's
mean [Fe/H] against $T_{\rm eff}$, for The Cannon (red) and ASPCAP (grey), with the slopes of straight-line fits.
Open clusters come out homogeneous at the level of a few hundredths of a dex in both methods (median 0.036 dex for
The Cannon, 0.034 dex for ASPCAP) — an upper limit on the precision, since the clusters may have a small intrinsic
spread and the membership is not perfect (the star at −0.34 dex is probably a non-member or a bad spectrum). In
the globular clusters The Cannon's spread is mostly *smaller* than ASPCAP's; part of that may be real precision,
but for M3, M2 and M13, at the lower edge of the training range, it is also the flattening of Chapter 5, which
compresses the labels near the edge. M71 is the exception, with a larger Cannon spread. Neither method shows a
strong trend of [Fe/H] with $T_{\rm eff}$ within the open clusters (slopes of a few hundredths of a dex per 1000 K).

## Summary

- Clusters test labels independently of the training set: the cluster means must follow ASPCAP across clusters,
  and the members of each open cluster must agree with each other.
- Inside the training range both tests pass; below it (metal-poor globulars) The Cannon is biased.

**Exercises.**
1. Add the metal-poor globular clusters to the training set (with the cluster mean [Fe/H] as label for all their
   members, as N15 did). Do the field metal-poor giants of Chapter 5 improve?
2. Compute the intrinsic spread of [Fe/H] in M67 by maximum likelihood, using the single-visit precision of
   Chapter 6 (scaled to each star's S/N) as the measurement error.